# Problem Set 3

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split

np.random.seed(42)

## Exercise 3.1: Ridge Regression

As demonstrated in the lecture on {ref}`sec:regression`, a machine learning model can overfit when its complexity is too high compared with the amount or structure of the data. Methods that reduce this tendency are called **regularisation**. In regression, one common example is **ridge regression** (or L2-regularisation) which extends the ordinary least squares loss function {ref}`eq:lsq-loss` by an additional penalty term:

:::{math}
:enumerated: true
:label: eq:ridge-loss
\mathcal{L}(\beta) = \frac{1}{N} \sum_{i=1}^N (y_i - \hat{y}_i)^2 + \lambda \|\beta\|_2^2
:::

The second term penalises large weights, so the model cannot fit the data at any cost; it has to *pay* for every large coefficient. The regularisation parameter $\lambda$ controls the trade-off between fitting the data and keeping the weights small.

**(a) Implement the ridge regression loss for polynomial regression in one dimension.**<br>

In [ ]:
def ridge_loss(beta: np.ndarray, x: np.ndarray, y: np.ndarray, lam: float) -> float:
# your code here





    return loss

**(b) Fit the non-regularised and the regularised polynomial models on the methylene blue data from the {ref}`sec:regression` section.**<br>
*Hint: To use the code below, the training data must be stored in the arrays `x_train` and `y_train`. Make sure to standardise the features before fitting the models.*

In [ ]:
# your code here













In [ ]:
from scipy.optimize import minimize

deg = 15
lam = 0.01

# Fit the non-regularised model
beta_non_ridge = np.polyfit(x_train, y_train, deg)

# Fit the regularised model
result = minimize(
    ridge_loss,
    beta_non_ridge,
    args=(x_train, y_train, lam),
    method="BFGS",
    options={"maxiter": 1000}
)

if not result.success:
    print(f"Warning: optimizer did not converge ({result.message}).")

beta_ridge = result.x

**(c) Compute the mean squared error of the non-regularised and the regularised models on the training and test data, respectively. How can you detect overfitting from these values?**<br>

In [ ]:
# your code here













**(d) Plot the training and test data, as well as the non-regularised and the regularised polynomials.**<br>

In [ ]:
# your code here













:::{note} Multicollinearity in linear regression

The example above, where polynomial regression is applied to a perfectly linear one-dimensional dataset, is intentionally contrived. However, it provides a simple setting for illustrating overfitting and regularisation. In real-world applications, ridge regression is often used for high-dimensional linear regression, especially when features are strongly correlated (multicollinearity). This is an important consideration when engineering features for machine learning models.

:::

## Exercise 3.2: Kernel SVM 

Kernel SVMs address {ref}`sec:classification` problems where the data is not linearly separable in the original feature space. The key idea is to map the data to a higher-dimensional space in which a linear separation becomes possible. To illustrate this idea, we use the artificial dataset shown below, generated with the [`make_circles`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_circles.html) function from the [`scikit-learn`](https://scikit-learn.org/stable/) library.

In [ ]:
# Reuse the simple linear SVM from the classification chapter.
# The bias is the feature x_0 = 1, so beta has length dim + 1 and labels are -1/+1.
class SupportVectorMachine:
    def __init__(self, dim=2, alpha=0.1, n_epochs=100, lam=0.1):
        self.alpha = alpha
        self.n_epochs = n_epochs
        self.lam = lam
        self.beta = np.random.randn(dim + 1)

    def net_input(self, x):
        return x @ self.beta

    def fit(self, X, y):
        N = len(X)
        for _ in range(self.n_epochs):
            for xi, yi in zip(X, y):
                update = yi if yi * self.net_input(xi) < 1.0 else 0.0
                self.beta += self.alpha * (update * xi / N - self.lam * self.beta)
        return self

    def predict(self, x):
        return np.where(self.net_input(x) >= 0.0, 1, -1)

In [ ]:
from sklearn.datasets import make_circles

# Two concentric circles in 2D: not linearly separable in the original space.
X, y = make_circles(n_samples=200, factor=0.3, noise=0.08, random_state=0)
print(X.shape)

# inner circle -> +1, outer ring -> -1
y = np.where(y == 1, 1, -1)

plt.plot(X[y == 1, 0], X[y == 1, 1], 'o', color='tab:red', label='Label: +1')
plt.plot(X[y == -1, 0], X[y == -1, 1], 'o', color='tab:blue', label='Label: -1')
plt.xlabel('$x_1$')
plt.ylabel('$x_2$')
plt.legend()
plt.show()

For a 2D dataset, this mapping can be defined through a non-linear feature map $\phi$:

$$
(x_1, x_2) \rightarrow (x_1, x_2, \phi(x_1, x_2))
$$

**(a) Construct a suitable feature map $\phi$ that allows to separate the two classes in the transformed three-dimensional space. Then, fit a SVM to the transformed data and compute the accuracy on the test set.**<br>

In [ ]:
# your code here


















**(b) Plot the data and the fitted hyperplane in three dimensions.**<br>
*Hint: You can find some examples of how to plot points and surfaces in three dimensions [here](https://matplotlib.org/stable/gallery/mplot3d/index.html).*

In [ ]:
# your code here






















## Exercise 3.2: $k$-Means Clustering

Revisit the section on {ref}`sec:clustering`. A measure of how well an algorithm identifies clusters in the data is the **cluster energy**

$$
E = \sum_{k=1}^K \sum_{\vec{x}_i \in C_k} \|x_i - m_k\|^2_2,
$$

which is minimised by the $k$-means algorithm. However, the number of clusters $K$ is a parameter of the algorithm and must be specified beforehand.

**(a) For which value of $K$ does the cluster energy $E$ reach its minimum? Why is this not a useful solution, and how is it related to overfitting?**<br>